In [69]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

### LOGISTIC REGRESSION

In [70]:
preprocessed_df = pd.read_csv('../data/datasets/preprocessed_dataset.csv')

In [71]:
preprocessed_df = preprocessed_df.copy()

In [72]:
# Columns with NA values
na_columns = preprocessed_df.isna().sum()[preprocessed_df.isna().sum() > 0]
print("Columns with NA values:")
print(na_columns)
print()

# Print each row that contains at least one NA
na_rows = preprocessed_df[preprocessed_df.isna().any(axis=1)]
print(f"Number of rows with NA: {len(na_rows)}")
print()
for idx, row in na_rows.iterrows():
    print(f"Index {idx}: {row.to_dict()}")


Columns with NA values:
lemmatized_text    33
dtype: int64

Number of rows with NA: 33

Index 32820: {'text': 'For More On HIAS: Refugee Resettlement Watch', 'label': 0, 'cleaned_text': 'For More On HIAS: Refugee Resettlement Watch', 'lemmatized_text': nan}
Index 32837: {'text': 'HE WILL NOT DIVIDE US .', 'label': 0, 'cleaned_text': 'HE WILL NOT DIVIDE US .', 'lemmatized_text': nan}
Index 32879: {'text': 'Via: NPR', 'label': 0, 'cleaned_text': 'Via: NPR', 'lemmatized_text': nan}
Index 32976: {'text': 'Via: MRCTV', 'label': 0, 'cleaned_text': 'Via: MRCTV', 'lemmatized_text': nan}
Index 33002: {'text': 'Via: WT', 'label': 0, 'cleaned_text': 'Via: WT', 'lemmatized_text': nan}
Index 33055: {'text': 'pic.twitter.com/KMnLrwB6t1  Richard K. Jones (@butlersheriff) December 19, 2016', 'label': 0, 'cleaned_text': 'Richard K. Jones ( ) December', 'lemmatized_text': nan}
Index 33103: {'text': 'DONALD TRUMP and KANYE WEST Greet Reporters After Meeting in Trump Tower ! KANYE WEST Jumped on the Trump

In [73]:
preprocessed_df = preprocessed_df.dropna()
preprocessed_df.isna().sum()

text               0
label              0
cleaned_text       0
lemmatized_text    0
dtype: int64

In [74]:
preprocessed_df.isna().sum()[preprocessed_df.isna().sum() > 0]

Series([], dtype: int64)

In [75]:
data_to_train = preprocessed_df[["lemmatized_text", "label"]].copy()
text_data = data_to_train["lemmatized_text"]
labels = data_to_train["label"]
X_train, X_test, y_train, y_test = train_test_split(text_data, labels, test_size=0.2, random_state=42, stratify=labels)

In [76]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidt = vectorizer.transform(X_test)

In [77]:
logistic_regression_model = LogisticRegression()
logistic_regression_model.fit(X_train_tfidf, y_train)
features_name = vectorizer.get_feature_names_out()
coefs = logistic_regression_model.coef_
coefs[0]

array([-0.00035537, -0.00073793, -0.25734086, ...,  0.09899258,
       -0.01094434, -0.03276764], shape=(36958,))

In [78]:
top_15_false_index = coefs[0].argsort()[0:25]

for index in top_15_false_index:
    print(f"{coefs[0][index]:.4f}: {features_name[index]}")

-4.9907: like
-4.5721: explain
-4.5084: story
-4.4011: reveal
-4.0026: watch
-3.9776: claim
-3.8723: thing
-3.8687: video
-3.8463: fact
-3.8158: racist
-3.5930: reason
-3.5644: press
-3.4810: reportedly
-3.4166: course
-3.3199: wonder
-3.2019: guy
-3.2018: lie
-3.1933: exactly
-3.1663: liberal
-3.1027: feature
-3.0700: know
-2.9907: recently
-2.9867: accord
-2.9692: way
-2.9415: apparently


In [79]:
top_15_true_index = coefs[0].argsort()[-25:]

for index in top_15_true_index:
    print(f"{coefs[0][index]:.4f}: {features_name[index]}")

2.7190: businessman
2.8346: agency
2.8452: independence
2.8967: add
2.9052: lawmaker
2.9239: region
2.9639: chief
2.9775: comment
3.1215: seek
3.2357: urge
3.2476: edit
3.3792: cite
3.4639: british
3.5565: leader
3.8813: rival
3.9088: parliament
3.9327: spokeswoman
3.9709: presidential
4.0797: militant
4.1308: republican
4.4614: minister
4.4943: reporter
4.7518: statement
5.6009: spokesman
8.1391: say


In [80]:
from sklearn.metrics import classification_report

y_pred = logistic_regression_model.predict(X_test_tfidt)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.86      0.87      4180
           1       0.89      0.90      0.90      5138

    accuracy                           0.88      9318
   macro avg       0.88      0.88      0.88      9318
weighted avg       0.88      0.88      0.88      9318



In [92]:
test_text = "The government has announced a new policy to combat climate change."
test_text_tfidf = vectorizer.transform([test_text])
prediction = logistic_regression_model.predict_proba(test_text_tfidf)
print(f"Prediction for test text: {prediction[0]}")

Prediction for test text: [0.80639554 0.19360446]
